<a href="https://colab.research.google.com/github/matjesg/deepflash2/blob/colab-test-modernize/colab_eval_encoders.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# deepflash2 — encoder benchmark: ConvNeXt V2 vs MaxViT vs ResNet

Cross-validates on the sample data and compares **validation Dice** across encoders. Only the encoder changes — the U-Net decoder, ensemble, and uncertainty stack are identical:
- **`tu-convnextv2_tiny.fcmae_ft_in22k_in1k`** — current default (ConvNet, SSL-pretrained)
- **`tu-maxvit_tiny_tf_512.in1k`** — transformer(-hybrid) backbone
- **`tu-resnet34`** — plain-CNN baseline

Same folds (`random_state=42`); each encoder trained over `N_FOLDS` splits, Dice reported as mean ± std.

**Before running:** `Runtime` → `Change runtime type` → **GPU**, then `Runtime` → `Run all`.

> Caveats: only 5 sample images — indicative, not a benchmark. Transformers are data-hungry, so **MaxViT may well *not* beat ConvNeXt V2** in this small-data regime — that's the honest expectation we're testing. Runtime ≈ `N_FOLDS × 3 encoders`; expect ~20–40 min on a Colab T4 at tile 256.

In [ ]:
#@title 1. Install deepflash2 (transformer-encoder branch) + force the modern fastai/fastcore pair, then check GPU
#@markdown If you already ran this in this session and hit an error, do **Runtime → Restart session** first (Colab preloads an old fastai that stays cached in the kernel).
!pip install -Uqq "git+https://github.com/matjesg/deepflash2.git@feat/transformer-encoders" "fastai>=2.8" "fastcore>=2.0"
from IPython.display import clear_output
from packaging.version import parse
import torch, fastai, fastcore, deepflash2
clear_output(wait=True)
print('deepflash2', deepflash2.__version__, '| torch', torch.__version__, '| cuda:', torch.cuda.is_available())
print('fastai', fastai.__version__, '| fastcore', fastcore.__version__)
assert torch.cuda.is_available(), 'Select a GPU runtime: Runtime -> Change runtime type -> GPU'
assert parse(fastai.__version__) >= parse('2.8'), (
    f'Stale fastai {fastai.__version__} is loaded (Colab preloaded an old one). '
    'Do Runtime -> Restart session, then Run all again.')
print('OK: modern fastai/fastcore loaded')

In [ ]:
#@title 2. Download sample training data (5 image/mask pairs)
from deepflash2.utils import download_sample_data
download_sample_data('https://github.com/matjesg/deepflash2/releases/download/sample_data/',
                     'wue1_cFOS_small.zip', 'deepflash2_sample_data', extract=True)

In [ ]:
#@title 3. Cross-validate each encoder and compare validation Dice
import warnings; warnings.filterwarnings('ignore')
import gc, numpy as np, torch, pandas as pd
from deepflash2.config import Config
from deepflash2.learner import EnsembleLearner

DATA = 'deepflash2_sample_data'
N_FOLDS    = 3    #@param {type:'integer'}  folds to average over (max 5 for the sample data)
N_EPOCHS   = 15   #@param {type:'integer'}  epochs per fold
TILE_SHAPE = 256  #@param {type:'integer'}  training tile size (512 = MaxViT native but ~4x slower)

ENCODERS = {
    'convnextv2_tiny (default)': 'tu-convnextv2_tiny.fcmae_ft_in22k_in1k',
    'maxvit_tiny (transformer)': 'tu-maxvit_tiny_tf_512.in1k',
    'resnet34 (baseline)':       'tu-resnet34',
}

def eval_encoder(enc):
    cfg = Config(encoder_name=enc, n_epochs=N_EPOCHS, tile_shape=TILE_SHAPE)  # random_state=42 -> identical folds
    el = EnsembleLearner(image_dir='images', mask_dir='masks', path=DATA, config=cfg)
    dices = []
    for fold in list(el.splits)[:N_FOLDS]:
        el.fit(fold)
        rec = el.recorder[fold]
        di = list(rec.metric_names).index('dice') - 1   # values rows exclude the 'epoch' column
        dices.append(max(float(v[di]) for v in rec.values))
    del el; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return dices

rows = []
for label, enc in ENCODERS.items():
    print(f'\n===== {label}: {enc} =====')
    try:
        d = eval_encoder(enc)
        print(f'  per-fold val dice: {[round(x, 4) for x in d]}')
        rows.append({'encoder': label, 'mean_val_dice': round(float(np.mean(d)), 4),
                     'std': round(float(np.std(d)), 4), 'folds': len(d)})
    except Exception as e:
        print(f'  FAILED: {type(e).__name__}: {e}')
        rows.append({'encoder': label, 'mean_val_dice': None, 'std': None, 'folds': 0})

df = pd.DataFrame(rows).sort_values('mean_val_dice', ascending=False, na_position='last')
print(f'\n===== Validation Dice, {N_FOLDS}-fold CV, {N_EPOCHS} epochs, tile {TILE_SHAPE} =====')
print(df.to_string(index=False))
df